In [ ]:
!pip install pandas numpy scikit-learn scipy statsmodels holidays openpyxl

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, log_loss, f1_score
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

import holidays
from itertools import combinations
from datetime import timedelta

In [6]:
uploaded = files.upload()

Saving fraudTest.csv to fraudTest (1).csv


In [7]:
arquivo = list(uploaded.keys())[0]
df = pd.read_csv(arquivo)
print("Arquivo carregado:", arquivo)
print("Dimensão da base:", df.shape)
df.head()

Arquivo carregado: fraudTest (1).csv
Dimensão da base: (555719, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0


In [ ]:
print("Colunas da base:")
for c in df.columns:
    print(c)

In [ ]:
df.info()

In [ ]:
df_model = df.copy()

# datas
df_model["trans_date_trans_time"] = pd.to_datetime(df_model["trans_date_trans_time"], errors="coerce")
df_model["dob"] = pd.to_datetime(df_model["dob"], errors="coerce")

# variável alvo
df_model["is_fraud"] = pd.to_numeric(df_model["is_fraud"], errors="coerce")

df_model.head(3)

In [ ]:
df_model["trans_hour"] = df_model["trans_date_trans_time"].dt.hour
df_model["trans_dayofweek"] = df_model["trans_date_trans_time"].dt.dayofweek
df_model["trans_month"] = df_model["trans_date_trans_time"].dt.month
df_model["trans_date"] = df_model["trans_date_trans_time"].dt.date

# idade
df_model["idade"] = ((df_model["trans_date_trans_time"] - df_model["dob"]).dt.days / 365.25).astype(float)

# after_midnight
df_model["after_midnight"] = (df_model["trans_hour"] < 6).astype(int)

# is_weekend
df_model["is_weekend"] = df_model["trans_dayofweek"].isin([5, 6]).astype(int)

df_model[["trans_hour", "trans_dayofweek", "trans_month", "idade", "after_midnight", "is_weekend"]].head()

In [12]:
us_holidays = holidays.US()

df_model["is_holiday"] = df_model["trans_date"].apply(lambda x: 1 if pd.notnull(x) and x in us_holidays else 0)

df_model[["trans_date", "is_holiday"]].head()

,trans_date,is_holiday
0,2020-06-21,0
1,2020-06-21,0
2,2020-06-21,0
3,2020-06-21,0
4,2020-06-21,0


In [13]:
df_model["dist_km_home"] = np.sqrt(
    (df_model["lat"] - df_model["merch_lat"])**2 +
    (df_model["long"] - df_model["merch_long"])**2
) * 111

In [14]:
df_model = df_model.sort_values(["cc_num", "trans_date_trans_time"]).reset_index(drop=True)

In [15]:
df_model["prev_lat"] = df_model.groupby("cc_num")["lat"].shift(1)
df_model["prev_long"] = df_model.groupby("cc_num")["long"].shift(1)

df_model["dist_km_last_transaction"] = np.sqrt(
    (df_model["lat"] - df_model["prev_lat"])**2 +
    (df_model["long"] - df_model["prev_long"])**2
) * 111

df_model["dist_km_last_transaction"] = df_model["dist_km_last_transaction"].fillna(0)

In [16]:
df_model["prev_time"] = df_model.groupby("cc_num")["trans_date_trans_time"].shift(1)
df_model["hours_since_last_trans"] = (
    (df_model["trans_date_trans_time"] - df_model["prev_time"]).dt.total_seconds() / 3600
)

df_model["hours_since_last_trans"] = df_model["hours_since_last_trans"].fillna(9999)

In [17]:
speed_kmh = df_model["dist_km_last_transaction"] / df_model["hours_since_last_trans"].replace(0, np.nan)
speed_kmh = speed_kmh.replace([np.inf, -np.inf], np.nan).fillna(0)

df_model["impossible_travel"] = (speed_kmh > 900).astype(int)

In [18]:
df_model["usr_avg_amt"] = df_model.groupby("cc_num")["amt"].transform("mean")
df_model["amt_over_usr_avg"] = df_model["amt"] - df_model["usr_avg_amt"]

In [19]:
df_model["first_time_merchant"] = ~df_model.duplicated(subset=["cc_num", "merchant"])
df_model["first_time_merchant"] = df_model["first_time_merchant"].astype(int)

In [20]:
def count_prev_24h(group):
    times = group["trans_date_trans_time"].values
    result = []
    for i, t in enumerate(times):
        start = t - np.timedelta64(24, 'h')
        count = np.sum((times[:i] >= start) & (times[:i] < t))
        result.append(count)
    return pd.Series(result, index=group.index)

df_model["n_trans_24h"] = df_model.groupby("cc_num", group_keys=False).apply(count_prev_24h)

In [21]:
def count_diff_merchants_7d(group):
    times = group["trans_date_trans_time"].values
    merchants = group["merchant"].values
    result = []
    for i, t in enumerate(times):
        start = t - np.timedelta64(7, 'D')
        mask = (times[:i] >= start) & (times[:i] < t)
        result.append(len(set(merchants[:i][mask])))
    return pd.Series(result, index=group.index)

df_model["num_diff_merchants_7d"] = df_model.groupby("cc_num", group_keys=False).apply(count_diff_merchants_7d)

In [22]:
df_model["new_zip_code"] = ~df_model.duplicated(subset=["cc_num", "zip"])
df_model["new_zip_code"] = df_model["new_zip_code"].astype(int)

In [23]:
def categorize_job(job):
    if pd.isna(job):
        return "Other"

    j = str(job).lower()

    if any(x in j for x in ["engineer", "developer", "programmer", "it", "analyst", "scientist"]):
        return "STEM"
    elif any(x in j for x in ["teacher", "professor", "librarian", "educat"]):
        return "Education"
    elif any(x in j for x in ["sales", "marketing", "retail"]):
        return "Sales"
    elif any(x in j for x in ["doctor", "nurse", "therapist", "psych", "health", "medical"]):
        return "Health"
    elif any(x in j for x in ["manager", "executive", "director", "administrator"]):
        return "Management"
    elif any(x in j for x in ["mechanic", "technician", "electrician", "repair", "worker", "driver"]):
        return "Technical_Manual"
    elif any(x in j for x in ["lawyer", "attorney", "judge", "legal"]):
        return "Legal"
    elif any(x in j for x in ["artist", "designer", "writer", "musician"]):
        return "Creative"
    else:
        return "Other"

df_model["job_type"] = df_model["job"].apply(categorize_job)

df_model[["job", "job_type"]].head(10)

,job,job_type
0,Information systems manager,Management
1,Information systems manager,Management
2,Information systems manager,Management
3,Information systems manager,Management
4,Information systems manager,Management
5,Information systems manager,Management
6,Information systems manager,Management
7,Information systems manager,Management
8,Information systems manager,Management
9,Information systems manager,Management


In [24]:
novas_features = [
    "after_midnight", "is_weekend", "is_holiday", "n_trans_24h",
    "first_time_merchant", "num_diff_merchants_7d", "new_zip_code",
    "dist_km_home", "dist_km_last_transaction", "impossible_travel",
    "amt_over_usr_avg", "job_type", "idade"
]

print("Novas features criadas:")
for c in novas_features:
    print(c)

Novas features criadas:
after_midnight
is_weekend
is_holiday
n_trans_24h
first_time_merchant
num_diff_merchants_7d
new_zip_code
dist_km_home
dist_km_last_transaction
impossible_travel
amt_over_usr_avg
job_type
idade


In [25]:
colunas_remover = [
    "first", "last", "street", "trans_num",
    "trans_date_trans_time", "dob", "prev_time",
    "prev_lat", "prev_long", "trans_date"
]

df_model = df_model.drop(columns=[c for c in colunas_remover if c in df_model.columns], errors="ignore")

In [26]:
target = "is_fraud"

X = df_model.drop(columns=[target])
y = df_model[target]

print("Dimensão X:", X.shape)
print("Dimensão y:", y.shape)
print("\nDistribuição do alvo:")
print(y.value_counts(dropna=False))

Dimensão X: (555719, 34)
Dimensão y: (555719,)

Distribuição do alvo:
is_fraud
0    553574
1      2145
Name: count, dtype: int64


In [27]:
categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()
numericas = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categóricas:", categoricas)
print("\nNuméricas:", numericas)

Categóricas: ['merchant', 'category', 'gender', 'city', 'state', 'job', 'job_type']

Numéricas: ['Unnamed: 0', 'cc_num', 'amt', 'zip', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long', 'trans_hour', 'trans_dayofweek', 'trans_month', 'idade', 'after_midnight', 'is_weekend', 'is_holiday', 'dist_km_home', 'dist_km_last_transaction', 'hours_since_last_trans', 'impossible_travel', 'usr_avg_amt', 'amt_over_usr_avg', 'first_time_merchant', 'n_trans_24h', 'num_diff_merchants_7d', 'new_zip_code']


In [28]:
transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessador = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, numericas),
        ("cat", transformador_categorico, categoricas)
    ]
)

In [29]:
def disparate_impact(df_slice, protected_col, privileged_value):
    df_valid = df_slice[[protected_col, "pred_class"]].dropna().copy()

    priv = df_valid[df_valid[protected_col] == privileged_value]
    unpriv = df_valid[df_valid[protected_col] != privileged_value]

    if len(priv) == 0 or len(unpriv) == 0:
        return np.nan

    p_priv = priv["pred_class"].mean()
    p_unpriv = unpriv["pred_class"].mean()

    if p_priv == 0:
        return np.nan

    return p_unpriv / p_priv

In [30]:
def calcular_fairness_agregada(df_slice):
    temp = df_slice.copy()

    # grupo etário
    if "idade" in temp.columns:
        temp["idade_grupo"] = pd.cut(
            temp["idade"],
            bins=[0, 30, 50, 120],
            labels=["jovem", "adulto", "maduro"]
        )
    else:
        temp["idade_grupo"] = np.nan

    di_genero = disparate_impact(temp, "gender", "M") if "gender" in temp.columns else np.nan
    di_job = disparate_impact(temp, "job_type", "STEM") if "job_type" in temp.columns else np.nan
    di_idade = disparate_impact(temp, "idade_grupo", "adulto")

    valores = [di_genero, di_job, di_idade]
    valores = [v for v in valores if pd.notnull(v)]

    if len(valores) == 0:
        return np.nan

    return np.mean(valores)

In [31]:
def fairness_score_from_di(di):
    if pd.isnull(di):
        return np.nan
    return max(0, 1 - abs(1 - di))

def objective_score(f1, di):
    fair = fairness_score_from_di(di)
    if pd.isnull(fair):
        return np.nan
    return f1 + fair

In [32]:
def criar_bins_para_slices(df_in):
    df_out = df_in.copy()

    bins_dict = {
        "amt": 5,
        "idade": 5,
        "city_pop": 5,
        "trans_hour": 4,
        "dist_km_home": 5,
        "dist_km_last_transaction": 5,
        "n_trans_24h": 5,
        "num_diff_merchants_7d": 5,
        "amt_over_usr_avg": 5
    }

    for col, bins in bins_dict.items():
        if col in df_out.columns:
            try:
                df_out[col + "_bin"] = pd.qcut(df_out[col], q=bins, duplicates="drop")
            except:
                df_out[col + "_bin"] = pd.cut(df_out[col], bins=bins)

    return df_out

In [47]:
def calcular_effect_size(loss_slice, loss_rest):
    var1 = np.var(loss_slice, ddof=1) if len(loss_slice) > 1 else 0
    var2 = np.var(loss_rest, ddof=1) if len(loss_rest) > 1 else 0
    denom = np.sqrt(var1 + var2)
    if denom == 0:
        return 0.0
    return np.sqrt(2) * (np.mean(loss_slice) - np.mean(loss_rest)) / denom


def avaliar_slice(df_base, filtro, descricao):
    slice_df = df_base[filtro].copy()
    rest_df = df_base[~filtro].copy()

    if len(slice_df) < 80 or len(rest_df) < 80:
        return None

    # performance
    try:
        f1_slice = f1_score(slice_df["is_fraud"], slice_df["pred_class"])
    except:
        f1_slice = np.nan

    # fairness
    di_slice = calcular_fairness_agregada(slice_df)
    obj_slice = objective_score(f1_slice, di_slice)

    # loss
    loss_slice = slice_df["loss"].dropna().values
    loss_rest = rest_df["loss"].dropna().values

    if len(loss_slice) < 2 or len(loss_rest) < 2:
        return None

    loss_slice_mean = loss_slice.mean()
    loss_rest_mean = loss_rest.mean()
    diff = loss_slice_mean - loss_rest_mean

    # queremos slice problemática: loss maior que o restante
    if diff <= 0:
        return None

    effect = calcular_effect_size(loss_slice, loss_rest)
    t_stat, p_value = ttest_ind(loss_slice, loss_rest, equal_var=False, alternative="greater")

    return {
        "slice": descricao,
        "size": len(slice_df),
        "slice_loss": loss_slice_mean,
        "rest_loss": loss_rest_mean,
        "diff_loss": diff,
        "effect_size": effect,
        "p_value": p_value,
        "f1_slice": f1_slice,
        "di_slice": di_slice,
        "objective_slice": obj_slice
    }

In [48]:
def rodar_experimento(df_model, experimento_id=1, test_size=0.30, random_state=42, T=0.40, k=20):
    X = df_model.drop(columns=["is_fraud"])
    y = df_model["is_fraud"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    categoricas = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    numericas = X_train.select_dtypes(include=[np.number]).columns.tolist()

    transformador_numerico = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    transformador_categorico = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessador = ColumnTransformer(
        transformers=[
            ("num", transformador_numerico, numericas),
            ("cat", transformador_categorico, categoricas)
        ]
    )

    modelo = Pipeline(steps=[
        ("preprocessador", preprocessador),
        ("classificador", RandomForestClassifier(
            n_estimators=200,
            random_state=random_state,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    modelo.fit(X_train, y_train)

    proba_test = modelo.predict_proba(X_test)[:, 1]
    pred_test = (proba_test >= 0.5).astype(int)

    eps = 1e-15
    proba_test_clip = np.clip(proba_test, eps, 1 - eps)

    loss_individual = -(
        y_test * np.log(proba_test_clip) +
        (1 - y_test) * np.log(1 - proba_test_clip)
    )

    df_test = X_test.copy()
    df_test["is_fraud"] = y_test.values
    df_test["pred_prob"] = proba_test
    df_test["pred_class"] = pred_test
    df_test["loss"] = loss_individual.values

    f1_global = f1_score(y_test, pred_test)
    di_global = calcular_fairness_agregada(df_test)
    objective_global = objective_score(f1_global, di_global)

    df_slices = criar_bins_para_slices(df_test)

    candidatas = [
    "merchant",
    "category",
    "gender",
    "city",
    "state",
    "job_type",
    "after_midnight",
    "is_weekend",
    "is_holiday",
    "first_time_merchant",
    "new_zip_code",
    "impossible_travel",
    "amt_bin",
    "idade_bin",
    "city_pop_bin",
    "trans_hour_bin",
    "dist_km_home_bin",
    "dist_km_last_transaction_bin",
    "n_trans_24h_bin",
    "num_diff_merchants_7d_bin",
    "amt_over_usr_avg_bin"
]

    slice_cols = [c for c in candidatas if c in df_slices.columns]

    resultados = []

    # slices de 1 literal
    for col in slice_cols:
        valores = df_slices[col].dropna().unique()
        for val in valores:
            filtro = df_slices[col] == val
            descricao = f"{col} = {val}"
            res = avaliar_slice(df_slices, filtro, descricao)
            if res is not None:
                resultados.append(res)
                    # slices de 2 literais
    for col1, col2 in combinations(slice_cols, 2):
        vals1 = df_slices[col1].dropna().unique()
        vals2 = df_slices[col2].dropna().unique()

        # limitar valores para evitar explosão muito grande
        vals1 = vals1[:5]
        vals2 = vals2[:5]

        for v1 in vals1:
            for v2 in vals2:
                filtro = (df_slices[col1] == v1) & (df_slices[col2] == v2)
                descricao = f"{col1} = {v1} AND {col2} = {v2}"
                res = avaliar_slice(df_slices, filtro, descricao)
                if res is not None:
                    resultados.append(res)

    if len(resultados) == 0:
        return {
            "metricas_globais": pd.DataFrame([{
                "experimento": experimento_id,
                "f1_global": f1_global,
                "di_global": di_global,
                "objective_global": objective_global
            }]),
            "slices": pd.DataFrame()
        }

    df_results = pd.DataFrame(resultados)

    rej, pvals_corrigidos, _, _ = multipletests(df_results["p_value"], alpha=0.05, method="fdr_bh")
    df_results["p_adjusted"] = pvals_corrigidos
    df_results["significativa"] = rej

    df_problematicas = df_results[
        (df_results["effect_size"] >= T) &
        (df_results["significativa"] == True)
    ].copy()

    df_problematicas["delta_objective_vs_global"] = df_problematicas["objective_slice"] - objective_global
    df_problematicas["experimento"] = experimento_id

    df_problematicas = df_problematicas.sort_values(
        by=["delta_objective_vs_global", "size", "effect_size"],
        ascending=[False, False, False]
    ).head(k)

    metricas_globais = pd.DataFrame([{
        "experimento": experimento_id,
        "f1_global": f1_global,
        "di_global": di_global,
        "objective_global": objective_global
    }])

    return {
        "metricas_globais": metricas_globais,
        "slices": df_problematicas
    }

In [49]:
N = 1
T = 0.20
k = 30

lista_metricas = []
lista_slices = []

for i in range(N):
    resultado = rodar_experimento(
        df_model=df_model,
        experimento_id=i+1,
        test_size=0.30,
        random_state=42 + i,
        T=T,
        k=k
    )

    lista_metricas.append(resultado["metricas_globais"])

    if not resultado["slices"].empty:
        lista_slices.append(resultado["slices"])

df_metricas_globais = pd.concat(lista_metricas, ignore_index=True)

if len(lista_slices) > 0:
    df_slices_final = pd.concat(lista_slices, ignore_index=True)
else:
    df_slices_final = pd.DataFrame()

print("Experimentos concluídos.")
print(df_metricas_globais.head())

if not df_slices_final.empty:
    display(df_slices_final.head())

Experimentos concluídos.
   experimento  f1_global  di_global  objective_global
0            1   0.680328   1.329571          1.350757


,slice,size,slice_loss,rest_loss,diff_loss,effect_size,p_value,f1_slice,di_slice,objective_slice,p_adjusted,significativa,delta_objective_vs_global,experimento
0,category = shopping_net AND amt_over_usr_avg_b...,1868,0.063792,0.005394,0.058398,0.419254,3.151621e-45,0.891986,1.088559,1.803427,4.286204e-43,True,0.452670,1
1,"trans_hour_bin = (19.0, 23.0] AND amt_over_usr...",5795,0.037290,0.004923,0.032367,0.224137,3.924675e-40,0.886179,1.459284,1.426894,3.335974e-38,True,0.076138,1
2,"category = shopping_net AND amt_bin = (94.21, ...",2489,0.048072,0.005411,0.042660,0.337349,3.111174e-42,0.891986,1.510748,1.381238,3.525997e-40,True,0.030481,1
3,"amt_bin = (94.21, 16339.26] AND trans_hour_bin...",5667,0.039729,0.004863,0.034867,0.230907,1.018870e-40,0.882236,1.529923,1.352312,9.237754e-39,True,0.001556,1
4,category = misc_net AND trans_hour_bin = (19.0...,541,0.055507,0.005887,0.049619,0.369793,4.773522e-12,0.920354,1.718700,1.201654,8.431155e-11,True,-0.149103,1


In [42]:
df_metricas_globais.describe()

,experimento,f1_global,di_global,objective_global
count,1.0,1.000000,1.000000,1.000000
mean,1.0,0.680328,1.329571,1.350757
std,NaN,NaN,NaN,NaN
min,1.0,0.680328,1.329571,1.350757
25%,1.0,0.680328,1.329571,1.350757
50%,1.0,0.680328,1.329571,1.350757
75%,1.0,0.680328,1.329571,1.350757
max,1.0,0.680328,1.329571,1.350757


In [43]:
if not df_slices_final.empty:
    recorrencia = (
        df_slices_final.groupby("slice")
        .agg(
            frequencia=("slice", "count"),
            media_objective=("objective_slice", "mean"),
            media_delta_obj=("delta_objective_vs_global", "mean"),
            media_effect_size=("effect_size", "mean"),
            media_tamanho=("size", "mean")
        )
        .reset_index()
        .sort_values(by=["frequencia", "media_delta_obj"], ascending=[False, False])
    )

    display(recorrencia.head(20))
else:
    print("Nenhuma slice problemática encontrada.")

Nenhuma slice problemática encontrada.


In [44]:
if not df_slices_final.empty:
    recorrencia = (
        df_slices_final.groupby("slice")
        .agg(
            frequencia=("slice", "count"),
            media_objective=("objective_slice", "mean"),
            media_delta_obj=("delta_objective_vs_global", "mean"),
            media_effect_size=("effect_size", "mean"),
            media_tamanho=("size", "mean")
        )
        .reset_index()
        .sort_values(by=["frequencia", "media_delta_obj"], ascending=[False, False])
    )

    melhores = recorrencia[recorrencia["media_delta_obj"] > 0]

    print("Quantidade de slices com melhoria média da função objetivo:", len(melhores))
    display(melhores.head(10))
else:
    print("Nenhuma slice encontrada para análise.")

Nenhuma slice encontrada para análise.


In [45]:
df_metricas_globais.to_csv("metricas_globais_experimentos.csv", index=False)

if not df_slices_final.empty:
    df_slices_final.to_csv("slices_problematicas_todos_experimentos.csv", index=False)

    recorrencia = (
        df_slices_final.groupby("slice")
        .agg(
            frequencia=("slice", "count"),
            media_objective=("objective_slice", "mean"),
            media_delta_obj=("delta_objective_vs_global", "mean"),
            media_effect_size=("effect_size", "mean"),
            media_tamanho=("size", "mean")
        )
        .reset_index()
        .sort_values(by=["frequencia", "media_delta_obj"], ascending=[False, False])
    )

    recorrencia.to_csv("recorrencia_slices.csv", index=False)

print("Arquivos salvos.")

Arquivos salvos.


In [52]:
import os

files.download("metricas_globais_experimentos.csv")

if not df_slices_final.empty:
    if os.path.exists("slices_problematicas_todos_experimentos.csv"):
        files.download("slices_problematicas_todos_experimentos.csv")
    else:
        print("Arquivo 'slices_problematicas_todos_experimentos.csv' não encontrado para download.")
    if os.path.exists("recorrencia_slices.csv"):
        files.download("recorrencia_slices.csv")
    else:
        print("Arquivo 'recorrencia_slices.csv' não encontrado para download.")
else:
    print("df_slices_final está vazia, não há arquivos de slices para download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Arquivo 'slices_problematicas_todos_experimentos.csv' não encontrado para download.
Arquivo 'recorrencia_slices.csv' não encontrado para download.
